### Overfiting is possible but giving a try is not bad :)

In [101]:
import torch
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import pandas as pd

In [102]:
df = pd.read_csv("train_dataset.csv")

In [103]:
len(df.columns) - 1 # parameters excluding overflow

5

In [104]:
df.head()

,flow_rate_L_min,concentration_mol_L,inlet_temperature_K,length_m,jacket_temperature_K,overall_yield
0,33.09,3.68,357.75,19.87,383.79,63.024
1,76.30,1.34,429.70,14.84,405.72,86.611
2,59.90,1.01,431.10,11.76,385.40,86.347
3,49.90,2.21,445.61,22.85,367.74,92.175
4,16.70,3.95,458.91,4.56,374.13,82.211


In [105]:
IN_FEATURES = len(df.columns) - 1
OUT_FEATURES = 1

In [106]:
# Converting data to tensors
# can't modify the data straight away hence got the copy() there
# converting to numpy for standard scaler
df = df[df["overall_yield"] != 0]
X = df.drop(columns=["overall_yield"]).to_numpy()
y = df["overall_yield"].to_numpy()

In [107]:
X.shape, y.shape

((113, 5), (113,))

In [108]:
# # train test split :)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)
X_train.shape

(90, 5)

In [109]:
from sklearn.preprocessing import StandardScaler

# Fit scaler ONLY on training data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [110]:
from torch import nn

class Overall_yield(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.stack = nn.Sequential(
            nn.Linear(in_features=IN_FEATURES, out_features=8),
            nn.Sigmoid(),
            # nn.Linear(in_features=8, out_features=8),
            # nn.Sigmoid(),
            nn.Linear(in_features=8, out_features=OUT_FEATURES),
        )

    def forward(self, x):
        return self.stack(x)

model = Overall_yield()
model

Overall_yield(
  (stack): Sequential(
    (0): Linear(in_features=5, out_features=8, bias=True)
    (1): Sigmoid()
    (2): Linear(in_features=8, out_features=1, bias=True)
  )
)

In [111]:
# Two options for loss -> Regression problem
# loss_fn = torch.nn.MSELoss()
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=0.01)

In [112]:
# testing if model works
# squeeze is required :)
model(X_train)[:5].squeeze()

tensor([-0.1162, -0.2361, -0.3744, -0.4645, -0.2633],
       grad_fn=<SqueezeBackward0>)

In [113]:
epochs = 8000

for epoch in range(epochs):
    # ----------------- Train -----------------
    model.train()

    train_pred = model(X_train)
    train_loss = loss_fn(train_pred.squeeze(), y_train)

    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()

    # ----------------- Test ------------------
    model.eval()

    with torch.inference_mode():
        test_pred = model(X_test)
        test_loss = loss_fn(test_pred.squeeze(), y_test)

    if epoch % 1000 == 0:
        print(
            f"Epoch: {epoch:4d} | "
            f"Train Loss: {train_loss.item():.4f} | "
            f"Test Loss: {test_loss.item():.4f}"
        )

Epoch:    0 | Train Loss: 3650.4597 | Test Loss: 3687.8613
Epoch: 1000 | Train Loss: 380.8403 | Test Loss: 777.0400
Epoch: 2000 | Train Loss: 355.7327 | Test Loss: 774.4830
Epoch: 3000 | Train Loss: 348.9202 | Test Loss: 781.2343
Epoch: 4000 | Train Loss: 345.3186 | Test Loss: 786.5414
Epoch: 5000 | Train Loss: 343.0369 | Test Loss: 789.8418
Epoch: 6000 | Train Loss: 341.4564 | Test Loss: 791.9180
Epoch: 7000 | Train Loss: 340.2931 | Test Loss: 793.3077


In [114]:
from sklearn.metrics import r2_score

model.eval()
with torch.inference_mode():
    pred = model(X_test).squeeze()

print(r2_score(y_test.numpy(), pred.numpy()))

0.40820932388305664
